# Frameworks 01 - OpenAI Agents

Objetivo: ejecutar agents.Runner real offline y demostrar Tools mixtas,
structured output, sessions, guardrails y handoffs mediante kwargs nativos.


**Lugar en el modelo:** OpenAI Agents es el Framework que controla Runner, Tools, sessions, guardrails y handoffs; no fija el Provider.

**Evidencia exigida:** el SDK real debe ejecutar su loop y conservar resultado nativo, adapter y Provider dentro del `RunResult` común.

**Límite de la evidencia:** con `python-runtime` se usa un modelo scripted para probar el Framework offline; los planes deterministas del System siguen perteneciendo a Agentic Systems.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_OPENAI_AGENTS_LIVE | 0 | Cambia Python por Provider auto. |
| Provider offline | python-runtime | Modelo scripted determinista. |
| Framework | openai-agents | Runner y agent loop reales. |


## 1) Frontera Provider x Framework


In [ ]:
import importlib.util
import os

import agentic_systems as toolkit

os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")

# OPENAI_AGENTS_DEPENDENCY: resolve availability and installation from the registry.
framework_name = "openai-agents"
install_target = toolkit.dependency_target(framework_name, kind="framework")
if install_target is None:
    raise RuntimeError(f"{framework_name!r} has no registered install target.")
if importlib.util.find_spec("agents") is None:
    get_ipython().run_line_magic("pip", f'install -q "{install_target}"')
importlib.invalidate_caches()
if importlib.util.find_spec("agents") is None:
    raise ImportError(
        f"{framework_name!r} remains unavailable after installing "
        f'"{install_target}". Restart this kernel and run the notebook again.'
    )

from agents import (
    GuardrailFunctionOutput,
    SQLiteSession,
    function_tool,
    handoff,
    input_guardrail,
)
from pydantic import BaseModel

import agentic_systems as toolkit

RUN_LIVE = os.getenv("RUN_OPENAI_AGENTS_LIVE", "0").strip().lower() in {"1", "true", "yes"}
runtime = (
    toolkit.runtime(
        provider="auto",
        provider_priority=["openai-runtime", "vllm-runtime", "bedrock-runtime"],
    )
    if RUN_LIVE
    else toolkit.runtime(provider="python-runtime")
)
profile = toolkit.integrations.framework_profile("openai-agents")
runtime_description = runtime.describe()
toolkit.show_json(
    {"runtime": runtime_description, "profile": profile.to_dict()},
    title="Provider x OpenAI Agents",
)


## 2) Tools mixtas, output tipado y session nativa


In [ ]:
class PublicEvidence(BaseModel):
    symbol: str
    is_public: bool

@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@function_tool
def native_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@input_guardrail(run_in_parallel=False)
def reject_blocked_input(context, native_agent, input_value):
    return GuardrailFunctionOutput(
        output_info={"checked": True},
        tripwire_triggered="blocked" in str(input_value).lower(),
    )

session = SQLiteSession("agentic-systems-tutorial", ":memory:")
framework = toolkit.framework(
    "openai-agents",
    agent_kwargs={
        "tools": [native_public_api],
        "input_guardrails": [reject_blocked_input],
        "output_type": PublicEvidence,
    },
    run_kwargs={"session": session},
)
agent = toolkit.agent(
    name="openai_agents_inspector",
    instructions="Usa la Tool solicitada y devuelve PublicEvidence.",
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    policy=toolkit.RunPolicy(max_turns=4, max_tool_calls=2),
)
agent.prepare()
result = await agent.arun(
    {"tool": "inspect_public_api", "input": {"symbol": "framework"}},
    mode="eval",
)
assert result.ok and result.data["is_public"]
assert result.engine == runtime_description["selected_provider"]
assert result.meta["framework_adapter"] == "openai-agents"
toolkit.human_result(result, title="OpenAI Agents RunResult", show_lineage=True)
toolkit.show_json(
    {
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(result.native_result).__name__,
        "framework_config": agent.framework_config.inspect(),
    },
    title="Runner evidence",
)


## 3) Guardrail operativo


In [ ]:
blocked_result = agent.run("blocked input", mode="eval")
assert not blocked_result.ok
session.close()
toolkit.show_json(
    {"ok": blocked_result.ok, "error": blocked_result.data["error"]["code"]},
    title="Input guardrail",
)


## 4) Handoff real entre dos agentes nativos


In [ ]:
specialist = toolkit.agent(
    name="specialist",
    instructions="Conserva la responsabilidad recibida.",
    runtime=runtime,
    tools=[inspect_public_api],
    framework="openai-agents",
)
specialist.prepare()
native_handoff = handoff(specialist.native_agent)

handoff_session = SQLiteSession("handoff-tutorial", ":memory:")

triage = toolkit.agent(
    name="triage",
    instructions="Transfiere al especialista cuando se solicite.",
    runtime=runtime,
    framework=toolkit.framework(
        "openai-agents",
        agent_kwargs={"handoffs": [native_handoff]},
        run_kwargs={"session": handoff_session},
    ),
)
handoff_result = triage.run(
    {"tool": native_handoff.tool_name, "input": {}},
    mode="eval",
)
assert handoff_result.ok
assert handoff_result.engine == runtime_description["selected_provider"]
assert handoff_result.meta["framework_adapter"] == "openai-agents"
handoff_session.close()
toolkit.show_json(
    {
        "handoff_tool": native_handoff.tool_name,
        "final_output": handoff_result.data,
        "native_result": type(handoff_result.native_result).__name__,
    },
    title="Native handoff",
)


## 5) API realmente ejercitada


In [ ]:
api_coverage = [
    "toolkit.dependency_target", "toolkit.runtime", "toolkit.framework",
    "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.RunPolicy",
    "toolkit.human_result", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="OpenAI Agents API coverage")


## Resultado e interpretacion

agents.Runner controla Tools, tipado, memoria, guardrails y handoff; el Provider
seleccionado controla modelo/transporte y Agentic Systems normaliza evidencia.
